In [31]:
import pandas as pd

# download from here: https://www.kaggle.com/datasets/stefanocoretta/yoda-speech-corpus?resource=download

FILE_PATH = r"-\yoda-corpus.csv"   # Replace with your actual path
df = pd.read_csv(FILE_PATH)

COL_SPEAKER = "character"  
COL_SCENE   = "slug"  
COL_MOVIE   = "movie"
COL_TYPE    = "component"
COL_TEXT    = "text"

# Find the 10 scenes with the highest number of dialogue lines.
print("\n1")
top_scenes = df[COL_SCENE].value_counts().head(10)
print(top_scenes.to_string())

# Total dialogue lines per movie.
print("\n2")
lines_per_movie = (
    df.groupby(COL_MOVIE)
      .size()
      .sort_values(ascending=False)
      .rename("line_count")
)

print(lines_per_movie.to_string())

# Per-movie top speaker, then rank characters by "wins"
print("\n3")
counts = (
    df.groupby([COL_MOVIE, COL_SPEAKER])
      .size()
      .reset_index(name="line_count")
)

top_per_movie = (
    counts.sort_values([COL_MOVIE, "line_count"], ascending=[True, False])
          .drop_duplicates(COL_MOVIE)
)

print("Top speaker in each movie:")
print(top_per_movie.to_string(index=False))

wins = top_per_movie[COL_SPEAKER].value_counts().rename("movie_wins")
print("\nRanking by number of wins:")
print(wins.to_string())

# Action/character scene ratio per movie, sorted by ratio
print("\n4")
type_counts = (
    df.groupby([COL_MOVIE, COL_TYPE])
      .size()
      .unstack(fill_value=0)
)
print("Raw counts per movie / type:")
print(type_counts.to_string())

ratio = type_counts["action"] / type_counts["character"].where(
    type_counts["character"] > 0
)
ratio = ratio.sort_values(ascending=False).rename("action_per_character")

print("\nAction-to-character ratio, sorted high to low:")
print(ratio.to_string())

# Per-character peak movie, then rank by peak count
print("\n5")
counts = (
    df.groupby([COL_SPEAKER, COL_MOVIE])
      .size()
      .reset_index(name="line_count")
)

peak = (
    counts.sort_values("line_count", ascending=False)
          .drop_duplicates(COL_SPEAKER)
          .set_index(COL_SPEAKER)
          .rename(columns={COL_MOVIE: "peak_movie",
                           "line_count": "peak_line_count"})
          .sort_values("peak_line_count", ascending=False)
)

print("Each character's peak movie and line count:")
print(peak.to_string())


1
slug
50   INT YODA'S HOUSE                                    37
INT. TEMPLE OF THE JEDI - COUNCIL CHAMBERS - TWILIGHT    34
INT. TEMPLE OF THE JEDI - COUNCIL CHAMBERS - DAY         27
83 INT. CORUSCANT-JEDI TEMPLE-COUNCIL CHAMBERS-DAY       25
90 INT. KASHYYYK-HOLOGRAM AREA-DAY                       25
185 INT. CORUSCANT-JEDI TEMPLE-CONTROL CENTER-NIGHT      24
INT. PALACE OF THE JEDI - COUNCIL CHAMBERS - SUNSET      21
233 INT. ALDERAAN CRUISER-CONFERENCE ROOM                19
164 INT. ALDERAAN-STARCRUISER-HALLWAY-DAY                16
152 EXT. KASHYYYK-LAKE ON VILLAGE EDGE-DUSK              15

2
movie
3    237
1     97
6     37

3
Top speaker in each movie:
 movie character  line_count
     1      YODA          29
     3      YODA          61
     6  narrator          14

Ranking by number of wins:
character
YODA        2
narrator    1

4
Raw counts per movie / type:
component  action  character
movie                       
1              17         80
3              60        